# Axiom Data：从 Raw 到跨仓消费者的可运行教学路径

这份 Notebook 使用真正的 `axiom_data`、Research adapter、Engine/Core、Runtime 输入闸门和 UI 投影代码。**第 1–6 节使用 synthetic fixture，第 7 节单独读取真实调试记录**；两类证据分开展示，不证明供应商历史公开 vintage 或全市场覆盖。仓库设计仍以 `design/02_axiom_data.md` 和 `design/06_axiom_ui.md` 为准。

推荐在 `axiom-data` 目录执行，Python 3.11+；需安装 `requirements-local.lock`，并将 Data、Engine、Research、UI 的 `src` 加入 `PYTHONPATH`。示例仅写临时目录，最后清理，无网络和凭据调用。

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import runpy, sys
repo = Path.cwd()
if repo.name == 'notebooks': repo = repo.parent
assert (repo / 'src/axiom_data').is_dir(), '请从 axiom-data 或 notebooks 目录运行'
for package in ('src', '../axiom-engine/src', '../axiom-research/src', '../axiom-ui/src'):
    sys.path.insert(0, str((repo / package).resolve()))
example = runpy.run_path(str(repo / 'examples/local_walkthrough.py'))
completion = runpy.run_path(str(repo / 'examples/completion_walkthrough.py'))
temporary = TemporaryDirectory(prefix='axiom-notebook-')
root = Path(temporary.name)
print('临时根目录已创建；以下均为 synthetic 数据。')


临时根目录已创建；以下均为 synthetic 数据。


## 1. Raw → typed Parquet → Snapshot → Reader

先运行原来的最小路径。显式 `Data.update` 保存源响应 Raw，规范化为 Parquet，再原子发布不可变 Snapshot。对同一事实的重复观察保留 Raw，却不强制发布新 Snapshot；增加一个 session 会产生新版本。读取只按固定 Snapshot + QuerySpec 工作，不创建 View 文件。

In [2]:
small_data, small_snapshot, small_query, small_batch = example['run'](root / 'small')
small_manifest = small_data.store.load_snapshot(small_snapshot.snapshot_id)
print('Snapshot 域:', list(small_manifest['domains']))
print('本次查询字段来源:', small_batch.field_meta['close']['by_key'][0]['availability_basis'])
print('API 响应生成时间:', small_batch.to_response()['context']['generated_at'])


Synthetic facts (real Parquet / Reader path):
security_id    session  close
synthetic-A 2020-01-02   10.0
synthetic-B 2020-01-02   20.0
2020 operational visibility: ['not_visible_at_cutoff', 'not_visible_at_cutoff']
Duplicate fetch: Raw recorded; Snapshot unchanged.
New session: new Snapshot; old Snapshot values unchanged.
Query writes no View directory: True
Snapshot: s_2da33db425574ee72562ed6627de0d37db6357859d89c29df5ce2f8fa5e5e8c5
UI response keys: ['records', 'field_meta', 'context']
Snapshot 域: ['market_daily']
本次查询字段来源: declared_vendor_assumption
API 响应生成时间: 2026-09-28T13:52:24.419038+00:00


## 2. PIT 与价格派生：不要把探索假设当作历史公开证据

原例的 2020 价格在 2026 年才被本系统观察。`best_effort_vendor_v1` 可按显式来源假设用于探索，`operational_pit_v1` 不会倒填 2020 决策。复权时先用相同 Snapshot/PIT 截止选出价格和因子，再在一个窗口固定共同锚点；本 Notebook 的跨仓计算使用未复权价格，Runtime 回放也只能读取未复权行情。

In [3]:
from dataclasses import replace
strict = small_data.read(snapshot=small_snapshot.snapshot_id,
                         query=replace(small_query, pit_policy='operational_pit_v1'))
print('2020 operational 缺失原因:',
      [m['missing_reason'] for m in strict.field_meta['close']['by_key']])
assert all(m['missing_reason'] == 'not_visible_at_cutoff'
           for m in strict.field_meta['close']['by_key'])
adjustment = completion['adjustment_example'](root)
assert adjustment['values'] == [5.0, 5.0]


2020 operational 缺失原因: ['not_visible_at_cutoff', 'not_visible_at_cutoff']
同一锚点复权价: [5.0, 5.0] ；跨日收益: 0.0


## 3. 财务原生事件 → 单季 → TTM

`Data.events` 先在 knowledge cutoff 选修订，再按报告期范围投影。纯函数 `single_quarter` 从累计 YTD 求相邻差，`ttm` 要求连续四季；撤销某季后返回 null 与组件来源，而不是补零。这里的 100/200/300/400/120 是人为构造的 CNY 数值。

In [4]:
finance = completion['finance_example'](root)
assert finance['old_ttm'] == 420 and finance['new_missing_reason'] == 'retracted'


财务事件原生键: ['security_id', 'endpoint', 'report_type', 'report_period']
单季收入: [100, 100, 100, 100, 120] CNY
旧 Snapshot 最近 TTM: 420 CNY
撤销后最近 TTM: <NA> 原因: retracted


## 4. 状态诊断与研究读取范围

参考域明确给出交易日历、证券身份、停牌和指数成分区间。`Data.states` 是诊断网格：正常、停牌与已声明覆盖的缺价分开，绝不将缺价编造成零价。`Data.plan_scope` 计算先前开放交易日的 warmup、历史成分并集和池外持仓；它报告证据缺口，不代替研究者判断可用性。

In [5]:
data, snapshot, query = completion['reference_example'](root)
print('固定 Snapshot:', snapshot)


状态诊断: ['normal_trading', 'suspended', 'source_gap'] ；缺失价格不是零价
研究 scope: warmup= ['2024-01-01'] history union= ['synthetic-A', 'synthetic-B'] held= ['held-C']
研究 scope 缺口: []
固定 Snapshot: s_d66677e137e86e49b5db73c95a05f8779baa3ab33439575fbed43f2d9af3045d


## 5. 消费边界：Data → Research → Core；Runtime 与 UI 分开

Research adapter 将同一个 DataBatch 的值、单位、逐键来源、缺失与每个 session 的 cutoff 映射至 Core ABI，执行的 value/lag/return 由 **Core** 算出。Runtime 在决策 session 固定 Snapshot，分别记录 decision_facts 与 market_replay 的真实查询引用；label_outcomes 不能进入决策事实。陈旧价格只作估值标记，不写回行情。UI 将保存的 Data query refs 投影为 Data 层；Feature 产物未保存时明确显示 `ARTIFACT_NOT_FOUND`，不在浏览时重算。`generated_at` 是查询生成时间，和源观察时间不同。

In [6]:
consumer = completion['consumer_example'](data, snapshot, query)
assert consumer['chart']['session_refs']['2024-01-02']['snapshot_id'] == snapshot


Runtime: label_outcomes 被决策输入闸门拒绝
Core A 列 [value, lag, return]: [[10.0, None, None], [10.0, 10.0, 0.0], [None, 10.0, None]]
Core 输出单位: ['CNY/share', 'CNY/share', 'dimensionless']
Runtime 固定读引用: [('decision_facts', 's_d66677e137e86e49b5db73c95a05f8779baa3ab33439575fbed43f2d9af3045d'), ('decision_facts', 's_d66677e137e86e49b5db73c95a05f8779baa3ab33439575fbed43f2d9af3045d'), ('market_replay', 's_d66677e137e86e49b5db73c95a05f8779baa3ab33439575fbed43f2d9af3045d')]
旧价估值标记: {'price': 10.0, 'price_session': '2024-01-03', 'valuation_session': '2024-01-04', 'stale_sessions': 1, 'is_stale': True, 'source_ref': 'sha256:52f4623f5a1c2a62953b440ea345a712c24d1a601d46598902d380c085f95e8b', 'missing_reason': None}
UI Data 层: stage= canonical unit= CNY/share source freshness= {'last_source_available_time': '2024-01-03T10:00:00+00:00', 'last_observed_time': '2024-01-03T10:00:00+00:00'} generated_at= 2026-09-28T00:00:00Z
UI 缺失 Feature 层: ARTIFACT_NOT_FOUND model_input


## 6. 可移植恢复：固定闭包与离线重读

`export_bundle` 复制所选 Snapshot 的父链、Raw/Parquet 引用和当前工作树中的 Data 代码与依赖锁；`verify_bundle` 检查字节及闭包，`import_bundle` 在新根恢复。下面使用同一个 QuerySpec 验证恢复后结果一致。这个包是本地传输格式，不是来源真实性签名。

In [7]:
portable = completion['portable_example'](root, data, snapshot, query)
assert portable['snapshot_id'] == snapshot


Portable: SHA-256 closure verified; offline read matches; Snapshot= s_d66677e137e86e49b5db73c95a05f8779baa3ab33439575fbed43f2d9af3045d
Portable captured code files: 24


## 7. 真实三个月调试与后续接源

前面的教学输入是 synthetic；下面单独读取本地真实调试报告及固定 Snapshot。真实范围为
2026-06-01 至 2026-08-31，两只股票及沪深300指数；不是完整指数成分池。
数据取得之后，旧版本复读、同值日更、搬移和离线 Raw 重建分别验证。

新增供应商字段、迁移接口、新事件和供应商因子沿现有三层存储扩展，见
[接源演进说明](../docs/source-evolution.md)。本单元只读本机已有文件，不联网。

In [8]:
import json
from axiom_data import Data, QuerySpec
real_root = repo.parent / 'data/debug_three_months_202606_202608'
report_path = real_root / 'debug-report.json'
if report_path.is_file():
    report = json.loads(report_path.read_text())
    print('真实调试状态:', report['status'])
    print('域行数:', report.get('domains'))
    print('独立核对:', report.get('independent_checks'))
    if report['status'] == 'debug_chain_verified':
        sid = report['daily_refresh']['snapshot_id']
        days = ('2026-06-01', '2026-08-31')
        q = QuerySpec('market_daily', ('close', 'volume_shares'),
            ('debug-pingan', 'debug-pudong'), days, 'best_effort_vendor_v1',
            {day: day+'T23:00:00+08:00' for day in days})
        real = Data(real_root).read(snapshot=sid, query=q)
        print(real.frame.to_string(index=False))
        print('来源限制:', real.context['limitations'])
else:
    print('本机没有真实调试数据；上面的教学数据仍全部为 synthetic。')
reference_report = repo.parent / 'data/debug_reference_202606_202608/debug-report.json'
if reference_report.is_file():
    print('参考域独立调试:', json.dumps(json.loads(reference_report.read_text()), ensure_ascii=False, indent=2))


真实调试状态: debug_chain_verified
域行数: {'adjustment_factors': 130, 'benchmark_daily': 65, 'market_daily': 130, 'security_master': 2, 'security_status': 0, 'trading_calendar': 184}
独立核对: {'raw_to_canonical_prices_and_units': 130, 'missing_bars_on_supplier_open_calendar': 0, 'missing_open_cells': [], 'historical_operational_prices_visible': 0, 'scope': 'two debug bindings; calendar completeness relative to supplier only'}
 security_id    session  close  volume_shares
debug-pingan 2026-06-01  10.99       95459569
debug-pudong 2026-06-01   9.32       75072117
debug-pingan 2026-08-31  11.72       90885655
debug-pudong 2026-08-31   9.16       99682464
来源限制: ['vendor release time is an assumption, not historical vintage evidence', 'terminal states ordered by system observation under declared source policy; vendor revision/publication order is unknown']
参考域独立调试: {
  "status": "complete_observations",
  "raw_responses": 9,
  "catalogue_securities": 5555,
  "candidate_union_securities": 300,
  "obser

## 8. 从小样本走向历史 bulk

长历史任务先固定证券身份、日期和来源范围，按交易日批量请求全市场后筛选固定范围，再按月及请求上限生成候选 Snapshot；全部块成功才切换 `current`。失败的块从 Raw 续跑，已完成的块不重新下载。这里用 **1800 个虚构代码**只计算请求规模，不下载数据，也不代表 CSI1800 的真实历史成员。

成员准备见 [reference readiness](../docs/reference-readiness.md)：主表包括上市、退市和暂停上市切片，月度成分观测只给出候选并集和有证据的日期。全历史拉取耗时主要由请求数、供应商限速和文件写入决定；下方时间是限速下限，不是完成承诺。

默认 CLI 为 `prepare → plan → run → status → audit`，读取、重建、导出和导入走同一套当前 API。旧 publication / materialized View 代码完整保存在 `archive/legacy_v1`，不进入安装包和默认测试；历史测试的原有失败记录保留，归档不等于修复。

In [9]:
from axiom_data import plan_bulk_job, estimate_bulk_job
synthetic_codes = tuple(f'{i:06d}.SZ' for i in range(1, 1801))
job = plan_bulk_job(mode='bulk', symbols=synthetic_codes,
    start_session='2014-11-01', end_session='2026-09-28',
    identity_map={code: 'synthetic-'+code for code in synthetic_codes},
    endpoints=('trade_cal', 'daily', 'adj_factor', 'suspend_d'))
estimate = estimate_bulk_job(job, min_interval_seconds=0)
print('仅规划，无供应商调用:', json.dumps(estimate, ensure_ascii=False, indent=2))
assert estimate['max_requests_in_chunk'] <= job.max_requests_per_chunk


仅规划，无供应商调用: {
  "requests": 13052,
  "requests_lower_bound": 2,
  "request_count_basis": "calendar_day_upper_bound_before_calendar_fetch",
  "chunks": 144,
  "max_requests_in_chunk": 93,
  "max_selected_raw_responses_in_memory": 93,
  "by_endpoint": {
    "trade_cal": 2,
    "daily": 4350,
    "adj_factor": 4350,
    "suspend_d": 4350
  },
  "minimum_pacing_seconds": 0.2,
  "minimum_pacing_days": 2.3148148148148148e-06,
  "pacing_seconds_at_request_upper_bound": 2610.2000000000003,
  "pacing_note": "upper-bound dates; exact open-day count follows saved calendar; retries, cap splits and I/O excluded",
  "max_attempts_multiplier_excluded": true
}


## 9. 一年真实验收与十二年运行

本地交付入口为 `axiom-data prepare → plan → run → verify → audit`。一年配置使用 2025-09-01 至 2026-08-31 的固定 1800 标的，验证工程链路；十二年配置采用月度历史候选并集，不能把固定锚点成员用于无幸存者偏差的历史回测。

请求按交易日批量发出，存储按月整理；中断只需再次执行同一 `run --plan`。完成任务可以不带 token 离线重跑。`rebuild` 从 Raw 离线重建，`export / verify / import` 搬移 Snapshot 及其闭包。详细命令和配置见 [CLI](../docs/cli.md)。下方只读取已保存的验收报告，不发出网络请求。

In [10]:
delivery_report = repo / 'docs/delivery-validation.json'
if delivery_report.is_file():
    print(json.dumps(json.loads(delivery_report.read_text()), ensure_ascii=False, indent=2))
else:
    print('一年验收报告尚未生成。')


{
  "schema_version": "axiom_data_delivery_validation_v1",
  "date": "2026-09-28",
  "package_version": "0.1.0",
  "scope": {
    "start": "2025-09-01",
    "end": "2026-08-31",
    "anchor": "2026-08-31",
    "mode": "anchor_members",
    "securities": 1800,
    "historical_union_candidates": 1996,
    "trading_sessions": 242
  },
  "snapshot_id": "s_c3faabc9ec124597e91701af33ebac563cd386eff776ac467f2a2462340bff2b",
  "reference_requests": 47,
  "bulk": {
    "canonical_partitions": 61,
    "canonical_rows": 872654,
    "coverage_basis": "observed_responses_only",
    "plan_fingerprint": "e36dd3191e9dcbc250aeb7cdf220856431bcab3ca8cd5446d61b46988e39f2a5",
    "selected_raw_batches": 735,
    "selected_raw_rows": 2664283,
    "snapshot_id": "s_c3faabc9ec124597e91701af33ebac563cd386eff776ac467f2a2462340bff2b",
    "verified": true
  },
  "timing": {
    "bulk_active_process_seconds": 193.56,
    "bulk_wall_including_fix_pause_seconds": 282.616,
    "bulk_peak_rss_bytes": 327614464,
    "

## 10. 支持边界与清理

本演示完成的是 Data 语义及 Data-facing 消费协议。它没有完整 UI 前端、订单/账户引擎、正式回测或 Qlib 消费者。真实小窗口只证明该次来源响应和所检验的链路；它不证明全市场覆盖或历史公开 vintage。教学与真实输出分开，不将供应商未知状态补成确定事实。

In [11]:
temporary.cleanup()
print('临时数据已清理；未调用供应商网络。')


临时数据已清理；未调用供应商网络。
